In [ ]:
import re
import os
import time
import threading
import queue
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from watchdog.observers import Observer
# Import the 'Observer' class from the 'watchdog.observers' module.
# The Observer watches for file system events and dispatches them to event handlers.

from watchdog.events import FileSystemEventHandler
# Import the 'FileSystemEventHandler' class to create custom event handlers.

import tkinter as tk
from tkinter import simpledialog, messagebox
# Import Tkinter modules for creating graphical user interface dialogs.

# Define the regular expression pattern for the required file naming convention.
# The naming convention is: Name_Institute_Date_DataQualifier
# For example: 20241018_JFi_IPAT_DataSet1
pattern = re.compile(r'^\d{8}_[A-Za-z]+_[A-Za-z]+_[A-Za-z0-9]+$')
# This pattern matches names like "YYYYMMDD_Name_Institute_DataQualifier"

class FileNamingHandler(FileSystemEventHandler):
    # Create a subclass of 'FileSystemEventHandler' to define custom behavior when file system events occur.

    def __init__(self, event_queue):
        super().__init__()
        self.event_queue = event_queue

    def on_created(self, event):
        """
        This method is automatically called by the observer when a new file or directory is created in the monitored path.
        The 'event' parameter contains information about the event, such as the file path.
        """
        if not event.is_directory:
            # Add the file path to the queue
            self.event_queue(event.src_path)

def check_file_name(self, file_path):
    """
    Checks if the file name matches the required naming convention.
    If not, it prompts the user to rename the file.
    """
    filename = os.path.basename(file_path)
    base_name, extension = os.path.splitext(filename)

    if not pattern.match(base_name):
        # Use the regular expression 'pattern' to check if the file name matches the required naming convention.
        # If it doesn't match, proceed to prompt the user to rename the file.
        self.prompt_rename(file_path, filename)

def prompt_rename(self, file_path, filename):
    """
    Prompts the user to rename the file using a graphical dialog.
    If the user cancels the renaming process, move the file to a 'rename' folder.
    """
    root = tk.Tk()
    root.withdraw()  # Hide the root window.
    # Initialize Tkinter and hide the main window since we only need to display dialogs.

    # Prepare a warning message to inform the user about the invalid file name.
    message = (
        f"The file '{filename}' does not adhere to the naming convention.\n"
        f"The required naming format is: YYYYMMDD_Name_Institute_DataQualifier"
    )

    # Display a warning message box to the user.
    messagebox.showwarning("Invalid File Name", message)

    # Get the file extension.
    base_name, extension = os.path.splitext(filename)

    # Loop until the user provides a valid filename or cancels.
    new_name = base_name
    while True:
        # Prompt the user to enter a new file name, without changing the extension.
        new_base_name = simpledialog.askstring(
            "Rename File",
            "Enter a new file name (format: YYYYMMDD_Name_Institute_DataQualifier):",
            initialvalue=new_name
        )

        if not new_base_name:
            # If the user cancels the dialog, move the file to a 'rename' folder.
            dir_path = os.path.dirname(file_path)
            rename_folder = os.path.join(dir_path, 'rename')
            if not os.path.exists(rename_folder):
                try:
                    os.makedirs(rename_folder)
                except Exception as e:
                    messagebox.showerror("Error", f"Failed to create 'rename' folder: {e}")
                    return

            new_path = os.path.join(rename_folder, filename)
            # Handle the case where the file already exists in the 'rename' folder.
            counter = 1
            while os.path.exists(new_path):
                new_path = os.path.join(rename_folder, f"{base_name}_{counter}{extension}")
                counter += 1

            # Move the file to the 'rename' folder with a unique name.
            try:
                os.rename(file_path, new_path)
                # Inform the user that the file has been moved.
                messagebox.showinfo("File Moved", f"File renaming was cancelled. The file has been moved to: {new_path}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to move file: {e}")
            return

        new_name = f"{new_base_name}{extension}"

        if pattern.match(new_base_name):
            # If the new name matches the required pattern, proceed.
            break
        else:
            # Inform the user that the entered name is still invalid.
            messagebox.showwarning("Invalid File Name", "The new file name does not match the required format. Please try again.")

    dir_path = os.path.dirname(file_path)
    # Get the directory path of the file.

    new_path = os.path.join(dir_path, new_name)
    # Construct the full file path with the new file name.

    # Check if a file with the new name already exists
    if os.path.exists(new_path):
        counter = 1
        base_new_name, ext_new_name = os.path.splitext(new_name)
        while os.path.exists(new_path):
            new_path = os.path.join(dir_path, f"{base_new_name}_{counter}{ext_new_name}")
            counter += 1

    try:
        os.rename(file_path, new_path)
        # Inform the user that the file has been successfully renamed.
        messagebox.showinfo("Success", f"File renamed to '{os.path.basename(new_path)}'")
    except Exception as e:
        # If an error occurs during renaming, catch the exception.
        messagebox.showerror("Error", f"Failed to rename file: {e}")

def process_events():
    """
    Processes file events from the queue.
    This function runs in the main Tkinter thread.
    """
    while not event_queue.empty():
        file_path = event_queue.get()
        check_file_name(file_path)
    # Schedule the next check of the queue
    root.after(100, process_events)

if __name__ == "__main__":
    # This block will only execute if the script is run directly.

    root = tk.Tk()
    root.withdraw()  # Hide the root window.

    event_queue = queue.Queue()

    path_to_watch = r"D:/Monitored_Folders/SEM"
    # Specify the path of the directory to be monitored.

    event_handler = FileNamingHandler(event_queue)
    # Create an instance of the custom event handler.

    observer = Observer()
    # Create an Observer object to monitor the file system.

    observer.schedule(event_handler, path=path_to_watch, recursive=False)
    # Schedule the observer to monitor the specified path with the event handler.
    # Set 'recursive=True' to monitor subdirectories as well.

    # Start the observer in a separate thread.

    observer_thread = threading.Thread(observer.start())
    observer_thread.daemon = True # Set the thread as a daemon to stop it when the main thread exits.
    observer_thread.start()
    # Start the observer thread, which will now monitor the directory for events.

    print(f"Monitoring directory: {path_to_watch}")

    root.after(100, process_events)

    def on_closing():
        observer.stop()
        observer.join()
        root.destroy()
        print("Monitoring stopped.")

    root.protocol("WM_DELETE_WINDOW", on_closing)

    try:
        root.mainloop()
        # Start the Tkinter main event loop to display dialogs and process events.
    except KeyboardInterrupt:
        on_closing()


In [ ]:
import re
import os
import queue
import logging
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import tkinter as tk
from tkinter import simpledialog, messagebox

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define the regular expression pattern for the required file naming convention.
pattern = re.compile(r'^\d{8}_[A-Za-z]+_[A-Za-z]+_[A-Za-z0-9]+$')

class FileNamingHandler(FileSystemEventHandler):
    """
    Event handler that adds file creation events to a queue.
    """
    def __init__(self, event_queue):
        super().__init__()
        self.event_queue = event_queue

    def on_created(self, event):
        if not event.is_directory:
            # Add the file path to the queue
            self.event_queue.put(event.src_path)

class FileMonitorApp:
    """
    Main application class that monitors a directory and ensures files adhere to a naming convention.
    """
    def __init__(self, path_to_watch):
        self.path_to_watch = path_to_watch

        # Initialize Tkinter root
        self.root = tk.Tk()
        self.root.withdraw()  # Hide the root window

        # Initialize event queue
        self.event_queue = queue.Queue()

        # Set up the observer and event handler
        self.event_handler = FileNamingHandler(self.event_queue)
        self.observer = Observer()
        self.observer.schedule(self.event_handler, path=self.path_to_watch, recursive=False)
        self.observer.start()
        logging.info(f"Monitoring directory: {self.path_to_watch}")

        # Schedule the event processing method
        self.root.after(100, self.process_events)

        # Set up the window close protocol
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def check_file_name(self, file_path):
        """
        Checks if the file name matches the required naming convention.
        If not, prompts the user to rename the file.
        """
        filename = os.path.basename(file_path)
        base_name, extension = os.path.splitext(filename)

        if not pattern.match(base_name):
            logging.info(f"File '{filename}' does not match the naming convention.")
            self.prompt_rename(file_path, filename)

    def prompt_rename(self, file_path, filename):
        """
        Prompts the user to rename the file using a graphical dialog.
        If the user cancels, moves the file to a 'rename' folder.
        """
        message = (
            f"The file '{filename}' does not adhere to the naming convention.\n"
            f"The required naming format is: YYYYMMDD_Name_Institute_DataQualifier"
        )

        messagebox.showwarning("Invalid File Name", message)

        base_name, extension = os.path.splitext(filename)
        new_name = base_name

        while True:
            new_base_name = simpledialog.askstring(
                "Rename File",
                "Enter a new file name (format: YYYYMMDD_Name_Institute_DataQualifier):",
                initialvalue=new_name
            )

            if not new_base_name:
                # Move the file to the 'rename' folder
                self.move_to_rename_folder(file_path, filename)
                return

            new_name = f"{new_base_name}{extension}"

            if pattern.match(new_base_name):
                # Valid new name provided
                break
            else:
                messagebox.showwarning(
                    "Invalid File Name",
                    "The new file name does not match the required format. Please try again."
                )

                new_name = new_base_name

        self.rename_file(file_path, new_name)

    def move_to_rename_folder(self, file_path, filename):
        """
        Moves the file to a 'rename' folder if the user cancels the renaming process.
        """
        dir_path = os.path.dirname(file_path)
        rename_folder = os.path.join(dir_path, 'rename')

        if not os.path.exists(rename_folder):
            try:
                os.makedirs(rename_folder)
            except Exception as e:
                messagebox.showerror("Error", f"Failed to create 'rename' folder: {e}")
                return

        new_path = self.get_unique_path(rename_folder, filename)

        try:
            os.rename(file_path, new_path)
            messagebox.showinfo(
                "File Moved",
                f"File renaming was cancelled. The file has been moved to: {new_path}"
            )
            logging.info(f"File moved to '{new_path}'.")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to move file: {e}")
            logging.error(f"Failed to move file '{file_path}' to '{new_path}': {e}")

    def rename_file(self, file_path, new_name):
        """
        Renames the file to the new name provided by the user.
        """
        dir_path = os.path.dirname(file_path)
        new_path = self.get_unique_path(dir_path, new_name)

        try:
            os.rename(file_path, new_path)
            messagebox.showinfo("Success", f"File renamed to '{os.path.basename(new_path)}'")
            logging.info(f"File '{file_path}' renamed to '{new_path}'.")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to rename file: {e}")
            logging.error(f"Failed to rename file '{file_path}' to '{new_path}': {e}")

    def get_unique_path(self, directory, filename):
        """
        Generates a unique file path to avoid overwriting existing files.
        """
        base_name, extension = os.path.splitext(filename)
        counter = 1
        new_filename = filename
        new_path = os.path.join(directory, new_filename)

        while os.path.exists(new_path):
            new_filename = f"{base_name}_{counter}{extension}"
            new_path = os.path.join(directory, new_filename)
            counter += 1

        return new_path

    def process_events(self):
        """
        Processes file events from the queue.
        """
        while not self.event_queue.empty():
            file_path = self.event_queue.get()
            self.check_file_name(file_path)
        self.root.after(100, self.process_events)

    def on_closing(self):
        """
        Handles the application closing event.
        """
        self.observer.stop()
        self.observer.join()
        self.root.destroy()
        logging.info("Monitoring stopped.")

    def run(self):
        """
        Starts the Tkinter main loop.
        """
        try:
            self.root.mainloop()
        except KeyboardInterrupt:
            self.on_closing()

if __name__ == "__main__":
    path_to_watch = r"D:/Monitored_Folders/SEM"
    app = FileMonitorApp(path_to_watch)
    app.run()


In [ ]:
import re
import os
import queue
import logging
import threading
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import tkinter as tk
from tkinter import simpledialog, messagebox, ttk

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define the regular expression pattern for the required file naming convention.
pattern = re.compile(r'^\d{8}_[A-Za-z]+_[A-Za-z]+_[A-Za-z0-9]+$')

class FileNamingHandler(FileSystemEventHandler):
    """
    Event handler that adds file creation events to a queue.
    """
    def __init__(self, event_queue):
        super().__init__()
        self.event_queue = event_queue

    def on_created(self, event):
        if not event.is_directory:
            # Add the file path to the queue
            self.event_queue.put(event.src_path)

class BatchRenameDialog(tk.Toplevel):
    """
    Custom dialog for batch renaming files.
    """
    def __init__(self, parent, invalid_files, apply_to_all_callback):
        super().__init__(parent)
        self.title("Batch File Renamer")
        self.invalid_files = invalid_files
        self.apply_to_all_callback = apply_to_all_callback
        self.create_widgets()
        self.protocol("WM_DELETE_WINDOW", self.on_close)

    def create_widgets(self):
        tk.Label(self, text="The following files do not adhere to the naming convention:").pack(pady=5)

        self.tree = ttk.Treeview(self, columns=("Old Name", "New Name"), show='headings')
        self.tree.heading("Old Name", text="Old Name")
        self.tree.heading("New Name", text="New Name")
        self.tree.pack(fill=tk.BOTH, expand=True)

        for idx, file_info in enumerate(self.invalid_files):
            filename = os.path.basename(file_info['file_path'])
            self.tree.insert('', 'end', iid=idx, values=(filename, ""))

        btn_frame = tk.Frame(self)
        btn_frame.pack(pady=5)

        tk.Button(btn_frame, text="Rename Selected", command=self.rename_selected).pack(side=tk.LEFT, padx=5)
        tk.Button(btn_frame, text="Move All to 'rename' Folder", command=self.apply_to_all).pack(side=tk.LEFT, padx=5)

    def rename_selected(self):
        selected_items = self.tree.selection()
        for item_id in selected_items:
            idx = int(item_id)
            file_info = self.invalid_files[idx]
            filename = os.path.basename(file_info['file_path'])
            base_name, extension = os.path.splitext(filename)

            new_base_name = simpledialog.askstring(
                "Rename File",
                f"Enter a new file name for '{filename}' (format: YYYYMMDD_Name_Institute_DataQualifier):",
                initialvalue=base_name,
                parent=self
            )

            if new_base_name:
                if pattern.match(new_base_name):
                    new_name = f"{new_base_name}{extension}"
                    file_info['new_name'] = new_name
                    self.tree.set(item_id, column="New Name", value=new_name)
                else:
                    messagebox.showwarning(
                        "Invalid File Name",
                        f"The new file name '{new_base_name}' does not match the required format. Please try again.",
                        parent=self
                    )
            else:
                # User cancelled renaming; do nothing
                pass

    def apply_to_all(self):
        # Callback to move all remaining invalid files to 'rename' folder
        self.apply_to_all_callback()
        self.destroy()

    def on_close(self):
        # Close the dialog
        self.destroy()

class FileMonitorApp:
    """
    Main application class that monitors a directory and ensures files adhere to a naming convention.
    """
    def __init__(self, path_to_watch):
        self.path_to_watch = path_to_watch

        # Initialize Tkinter root
        self.root = tk.Tk()
        self.root.withdraw()  # Hide the root window

        # Initialize event queue
        self.event_queue = queue.Queue()

        # Initialize list to collect invalid files
        self.invalid_files = []

        # Set up the observer and event handler
        self.event_handler = FileNamingHandler(self.event_queue)
        self.observer = Observer()
        self.observer.schedule(self.event_handler, path=self.path_to_watch, recursive=False)
        self.observer.start()
        logging.info(f"Monitoring directory: {self.path_to_watch}")

        # Batch processing interval (in milliseconds)
        self.batch_interval = 5000  # 5 seconds

        # Schedule the event processing method
        self.root.after(100, self.process_events)

        # Set up the window close protocol
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def check_file_name(self, file_path):
        """
        Checks if the file name matches the required naming convention.
        If not, adds it to the invalid_files list.
        """
        filename = os.path.basename(file_path)
        base_name, extension = os.path.splitext(filename)

        if not pattern.match(base_name):
            logging.info(f"File '{filename}' does not match the naming convention.")
            self.invalid_files.append({'file_path': file_path, 'new_name': None})

    def process_events(self):
        """
        Processes file events from the queue and handles invalid files in batches.
        """
        while not self.event_queue.empty():
            file_path = self.event_queue.get()
            self.check_file_name(file_path)

        # If there are invalid files, show the batch dialog
        if self.invalid_files:
            self.root.after(0, self.show_batch_dialog)

        # Schedule the next batch processing
        self.root.after(self.batch_interval, self.process_events)

    def show_batch_dialog(self):
        """
        Displays the batch rename dialog to the user.
        """
        if hasattr(self, 'batch_dialog') and self.batch_dialog.winfo_exists():
            # Dialog is already open
            return

        self.batch_dialog = BatchRenameDialog(self.root, self.invalid_files, self.move_all_to_rename_folder)
        self.batch_dialog.grab_set()  # Make the dialog modal
        self.root.wait_window(self.batch_dialog)
        self.process_renames()

    def move_all_to_rename_folder(self):
        """
        Moves all invalid files to the 'rename' folder.
        """
        for file_info in self.invalid_files:
            if file_info['new_name'] is None:
                file_path = file_info['file_path']
                filename = os.path.basename(file_path)
                self.move_to_rename_folder(file_path, filename)
        self.invalid_files.clear()

    def process_renames(self):
        """
        Processes the renaming of files based on user input in the batch dialog.
        """
        for file_info in self.invalid_files[:]:  # Iterate over a copy of the list
            if file_info['new_name']:
                file_path = file_info['file_path']
                new_name = file_info['new_name']
                self.rename_file(file_path, new_name)
                self.invalid_files.remove(file_info)
            else:
                # If no new name was provided, move the file to 'rename' folder
                file_path = file_info['file_path']
                filename = os.path.basename(file_path)
                self.move_to_rename_folder(file_path, filename)
                self.invalid_files.remove(file_info)

    def move_to_rename_folder(self, file_path, filename):
        """
        Moves the file to a 'rename' folder if the user cancels the renaming process.
        """
        dir_path = os.path.dirname(file_path)
        rename_folder = os.path.join(dir_path, 'rename')

        if not os.path.exists(rename_folder):
            try:
                os.makedirs(rename_folder)
            except Exception as e:
                messagebox.showerror("Error", f"Failed to create 'rename' folder: {e}", parent=self.root)
                return

        new_path = self.get_unique_path(rename_folder, filename)

        try:
            os.rename(file_path, new_path)
            messagebox.showinfo(
                "File Moved",
                f"File '{filename}' has been moved to: {new_path}",
                parent=self.root
            )
            logging.info(f"File moved to '{new_path}'.")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to move file: {e}", parent=self.root)
            logging.error(f"Failed to move file '{file_path}' to '{new_path}': {e}")

    def rename_file(self, file_path, new_name):
        """
        Renames the file to the new name provided by the user.
        """
        dir_path = os.path.dirname(file_path)
        new_path = self.get_unique_path(dir_path, new_name)

        try:
            os.rename(file_path, new_path)
            messagebox.showinfo("Success", f"File renamed to '{os.path.basename(new_path)}'", parent=self.root)
            logging.info(f"File '{file_path}' renamed to '{new_path}'.")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to rename file: {e}", parent=self.root)
            logging.error(f"Failed to rename file '{file_path}' to '{new_path}': {e}")

    def get_unique_path(self, directory, filename):
        """
        Generates a unique file path to avoid overwriting existing files.
        """
        base_name, extension = os.path.splitext(filename)
        counter = 1
        new_filename = filename
        new_path = os.path.join(directory, new_filename)

        while os.path.exists(new_path):
            new_filename = f"{base_name}_{counter}{extension}"
            new_path = os.path.join(directory, new_filename)
            counter += 1

        return new_path

    def on_closing(self):
        """
        Handles the application closing event.
        """
        self.observer.stop()
        self.observer.join()
        self.root.destroy()
        logging.info("Monitoring stopped.")

    def run(self):
        """
        Starts the Tkinter main loop.
        """
        try:
            self.root.mainloop()
        except KeyboardInterrupt:
            self.on_closing()

if __name__ == "__main__":
    path_to_watch = r"D:/Monitored_Folders/SEM"
    app = FileMonitorApp(path_to_watch)
    app.run()
